# Seed 87 diagnosis (Person 1)

Seed 87's Stage A lowered the held-out pair AUC from 0.608 to 0.511 (the other seeds reach 0.735-0.759).
Three checks, about 25 minutes on a T4:

1. **Rerun Stage A of seed 87 exactly** (same commit `6278e43`, same settings). Collapses again -> reproducible,
   a genuine training instability. Normal AUC -> a non-deterministic or technical glitch.
2. **Stage A of seed 123** (a new seed, diagnostic only, not added to the five-seed results).
3. **The saved vectors**: are seed 87's usage vectors nearly identical (collapse)?

Settings: GPU T4 x2, Internet On. Inputs: your dataset `senseshift-synthetic`, **and** the output of the main-runs
notebook (Add Input > Notebooks > your main-runs notebook), for check 3.

In [ ]:
# Cell 1: find the synthetic English set in /kaggle/input
import os, glob, subprocess
OUT  = '/kaggle/working'
RUNS = f'{OUT}/runs_a3'            # run folders (with vectors/ for Person 2)
LOGS = f'{OUT}/gpu_logs'
CKPT = '/tmp/kg/checkpoints'   # Stage A checkpoints (~1.1 GB per seed, not kept in the output)
for d in (RUNS, LOGS, CKPT): os.makedirs(d, exist_ok=True)

# The dataset holds sentences.bin (= sentences.zip renamed, so Kaggle does not try to unpack it:
# Kaggle rejects 4 file names with an apostrophe). We unpack it here; Linux accepts those names.
found = [p for p in glob.glob('/kaggle/input/**/generated_sentences', recursive=True) if os.path.isdir(p)]
if not found:
    archives = [p for p in glob.glob('/kaggle/input/**/*', recursive=True)
                if os.path.isfile(p) and os.path.basename(p).startswith('sentences')]
    assert archives, 'sentences.bin not found: Add Input > your dataset'
    print('unpacking', archives[0], '(about 1 min)')
    !mkdir -p /tmp/kg/synthetic && unzip -q -o "{archives[0]}" -d /tmp/kg/synthetic
    found = [p for p in glob.glob('/tmp/kg/synthetic/**/generated_sentences', recursive=True) if os.path.isdir(p)]
GEN = found[0]
print('synthetic set:', GEN, '|', len(glob.glob(f'{GEN}/*/*.jsonl')), 'word files (about 2,345 expected)')

In [ ]:
# Cell 2: clone and check out exactly the commit the main runs used
%cd /kaggle/working
!rm -rf /tmp/kg/SenseShift
!git clone -q https://github.com/abiha25/SenseShift-Semantic-change-across-multiple-time-periods.git /tmp/kg/SenseShift
%cd /tmp/kg/SenseShift
!git checkout -q 6278e43 && git log --oneline -1
!pip install -q bcubed nltk pyyaml
!mkdir -p data/external/synthetic_en && ln -sfn "{GEN}" data/external/synthetic_en/generated_sentences
!git status --porcelain
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
# Cell 3: a script that runs ONLY Stage A, exactly as `python -m src.proposed run` does
script = '''
import sys, json
sys.path.insert(0, '.')
from src.baselines import setup                 # same set-up as the real run (config + seeded RNGs)
from src.proposed import train_stage_a
seed = int(sys.argv[1])
cfg, df, folds, gold1 = setup('configs/default.yaml', [f'seed={seed}', 'output.checkpoints_dir=/tmp/kg/diag_ckpt'])
hist = {}
train_stage_a(cfg, hist)
print('RESULT', json.dumps({'seed': seed, 'data': hist['stage_a']['data'], 'epochs': hist['stage_a']['epochs']}))
'''
open('/tmp/kg/stage_a_only.py', 'w').write(script)
import json
results = {}
def stage_a(seed):
    !rm -rf /tmp/kg/diag_ckpt
    out = !python /tmp/kg/stage_a_only.py {seed} 2>&1
    for line in out:
        if 'Stage A' in line: print(line)
    res = [json.loads(l[7:]) for l in out if l.startswith('RESULT ')]
    if not res: print('\n'.join(out[-30:])); return
    results[seed] = res[0]
    e = res[0]['epochs']
    print(f"seed {seed}: AUC before {e[0]['val']:.4f} -> after {e[-1]['val']:.4f}, loss {e[-1]['loss']:.4f}")

Original seed 87 (Kaggle main run): AUC 0.6077 -> **0.5112**, loss 0.0732, items_train 171,980.

In [ ]:
# Cell 4: check 1 - rerun Stage A of seed 87 (about 9 min)
stage_a(87)

For comparison: seeds 13, 42, 1234, 2027 reached 0.759, 0.735, 0.747, 0.736 (loss 0.059-0.061).

In [ ]:
# Cell 5: check 2 - Stage A of a new seed, 123 (about 9 min)
stage_a(123)
json.dump(results, open('/kaggle/working/seed87_diagnosis_stage_a.json', 'w'), indent=2)

In [ ]:
# Cell 6: check 3 - are seed 87's usage vectors collapsed?
import numpy as np, glob
rows = []
for d in sorted(glob.glob('/kaggle/input/**/runs_a3/*', recursive=True)):
    for f in sorted(glob.glob(f'{d}/vectors/fold*.npz'))[:10]:
        V = np.load(f)['V']; V = V / np.linalg.norm(V, axis=1, keepdims=True)
        rng = np.random.default_rng(0); i = rng.integers(0, len(V), 3000); j = rng.integers(0, len(V), 3000)
        cos = (V[i] * V[j]).sum(1)
        rows.append({'run': d.split('/')[-1][16:], 'fold': f.split('/')[-1][:-4],
                     'mean cos': round(float(cos.mean()), 4), 'sd cos': round(float(cos.std()), 4)})
import pandas as pd
if rows:
    t = pd.DataFrame(rows); t.to_csv('/kaggle/working/seed87_diagnosis_vectors.csv', index=False)
    display(t.groupby('run')[['mean cos', 'sd cos']].mean().round(4))
else:
    print('No vectors found: Add Input > Notebooks > your main-runs notebook (its Output)')